<a href="https://colab.research.google.com/github/ryanzh00/IMG2GPS/blob/main/IMG2GPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image2GPS: Predicting GPS Coordinates from Campus Images

This notebook trains a deep learning model to predict GPS coordinates from images taken around the University of Pennsylvania campus.

The model uses:
- ResNet50 pretrained on ImageNet
- Local XY coordinate transformation
- Haversine distance based loss
- Transfer learning and fine-tuning
- Data augmentation for improved generalization

The final evaluation metric is average Haversine distance in meters.


## Install Dependencies

Install required libraries for dataset loading, image processing, and model training.

In [ ]:
!pip install -q datasets geopy huggingface_hub pandas

## Import Libraries and Configure Training Environment

This section imports all required libraries and configures CUDA, mixed precision training, and reproducibility settings.

In [ ]:
from dataclasses import asdict, dataclass
import json
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from datasets import load_dataset
from PIL import Image

from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

EARTH_RADIUS_METERS = 6_371_000.0

torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

## Training Configuration

Define model architecture, image size, learning rate, batch size, and other hyperparameters used during training.

In [ ]:
DATASET_REPO = "ryanzho/img2gps"

BACKBONE = "resnet50"

IMAGE_SIZE = 384
BATCH_SIZE = 32

EPOCHS = 60

LR = 3e-4
WEIGHT_DECAY = 1e-4

DROPOUT = 0.2
METER_WEIGHT = 0.05

NUM_WORKERS = 10

OUTPUT_DIR = Path("artifacts/image2gps_better_model")

## Geographic Coordinate Utilities

The model predicts normalized local XY coordinates instead of raw latitude and longitude.

This section includes:
- Haversine distance calculations
- Latitude/longitude conversion functions
- Local coordinate transforms

In [ ]:
def haversine_distance_torch(lat1, lon1, lat2, lon2):
    lat1 = torch.deg2rad(lat1)
    lon1 = torch.deg2rad(lon1)
    lat2 = torch.deg2rad(lat2)
    lon2 = torch.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = torch.sin(dlat / 2.0) ** 2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2.0) ** 2
    a = torch.clamp(a, 0.0, 1.0 - 1e-7)
    c = 2.0 * torch.atan2(torch.sqrt(a), torch.sqrt(1.0 - a))
    return EARTH_RADIUS_METERS * c

def haversine_distance_np(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    a = np.clip(a, 0.0, 1.0 - 1e-7)

    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return EARTH_RADIUS_METERS * c

def latlon_to_local_xy_np(lat_deg, lon_deg, origin_lat_deg, origin_lon_deg):
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    origin_lat_rad = math.radians(origin_lat_deg)
    origin_lon_rad = math.radians(origin_lon_deg)
    x = (lon_rad - origin_lon_rad) * math.cos(origin_lat_rad) * EARTH_RADIUS_METERS
    y = (lat_rad - origin_lat_rad) * EARTH_RADIUS_METERS
    return x, y

def local_xy_to_latlon_torch(x_m, y_m, origin_lat_deg, origin_lon_deg):
    origin_lat = torch.tensor(origin_lat_deg, dtype=x_m.dtype, device=x_m.device)
    origin_lon = torch.tensor(origin_lon_deg, dtype=x_m.dtype, device=x_m.device)
    lat = origin_lat + torch.rad2deg(y_m / EARTH_RADIUS_METERS)
    lon = origin_lon + torch.rad2deg(x_m / (EARTH_RADIUS_METERS * math.cos(math.radians(origin_lat_deg))))
    return lat, lon

## Geographic Statistics

Training labels are normalized using statistics computed from the training split only.

In [ ]:
@dataclass
class GeoStats:
    origin_lat: float
    origin_lon: float
    x_mean: float
    x_std: float
    y_mean: float
    y_std: float

def compute_geo_stats_from_arrays(latitudes, longitudes):
    latitudes = np.asarray(latitudes, dtype=np.float64)
    longitudes = np.asarray(longitudes, dtype=np.float64)
    origin_lat = float(latitudes.mean())
    origin_lon = float(longitudes.mean())
    x_m, y_m = latlon_to_local_xy_np(latitudes, longitudes, origin_lat, origin_lon)

    return GeoStats(
        origin_lat=origin_lat,
        origin_lon=origin_lon,
        x_mean=float(x_m.mean()),
        x_std=float(x_m.std() + 1e-6),
        y_mean=float(y_m.mean()),
        y_std=float(y_m.std() + 1e-6),
    )

## Dataset Construction and Image Caching

This dataset class:
- Loads images from the Hugging Face dataset
- Converts GPS coordinates into normalized local XY coordinates
- Caches resized images in memory for faster training
- Applies augmentations during training

In [ ]:
class GPSImageDataset(Dataset):
    def __init__(self, rows, geo_stats, image_transform, cache_size=448):
        self.geo_stats = geo_stats
        self.image_transform = image_transform

        self.images = []
        self.targets_xy = []
        self.latlons = []

        cache_transform = transforms.Compose([
            transforms.Resize((cache_size, cache_size)),
            transforms.PILToTensor()
        ])

        print("Caching images...")
        for i in range(len(rows)):
            example = rows[i]

            image = example["image"]

            if isinstance(image, torch.Tensor):
                image = transforms.ToPILImage()(image)

            image = image.convert("RGB")
            image = cache_transform(image)

            lat = float(example["latitude"])
            lon = float(example["longitude"])

            x_m, y_m = latlon_to_local_xy_np(
                np.asarray(lat),
                np.asarray(lon),
                self.geo_stats.origin_lat,
                self.geo_stats.origin_lon
            )

            target_xy = torch.tensor([
                (float(x_m) - self.geo_stats.x_mean) / self.geo_stats.x_std,
                (float(y_m) - self.geo_stats.y_mean) / self.geo_stats.y_std
            ], dtype=torch.float32)

            latlon = torch.tensor([lat, lon], dtype=torch.float32)

            self.images.append(image)
            self.targets_xy.append(target_xy)
            self.latlons.append(latlon)

        print(f"Cached {len(self.images)} images.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        image = self.image_transform(image)
        return image, self.targets_xy[index], self.latlons[index]

## Model Architecture

The model uses a pretrained ResNet50 backbone with a custom regression head for predicting 2D coordinates.

Transfer learning is used to improve convergence and generalization.

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        weights = ResNet50_Weights.IMAGENET1K_V2
        model = models.resnet50(weights=weights)

        in_features = model.fc.in_features

        model.fc = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

        self.model = model

    def forward(self, x):
        return self.model(x)

## Data Augmentation

Training augmentations improve generalization and reduce overfitting by exposing the model to variations in:
- lighting
- blur
- grayscale conditions
- image crops
- erased regions

Validation and test preprocessing use deterministic transforms.

In [ ]:
def build_transforms(image_size):
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ColorJitter(0.15, 0.15, 0.15, 0.03),
        transforms.RandomGrayscale(p=0.1),   # 👈 HUGE for generalization
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),  # 👈 robustness
        transforms.ConvertImageDtype(torch.float32),
        transforms.RandomErasing(p=0.1),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.ConvertImageDtype(torch.float32),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
    ])

    return train_transform, eval_transform

## Blended Regression Loss

The model is trained using:
- Huber loss on normalized XY coordinates
- Haversine distance regularization in meters

This improves both numerical stability and geographic accuracy.

In [ ]:
def unnormalize_xy(pred_xy_norm, geo_stats):
    pred_x = pred_xy_norm[:, 0] * geo_stats.x_std + geo_stats.x_mean
    pred_y = pred_xy_norm[:, 1] * geo_stats.y_std + geo_stats.y_mean
    return pred_x, pred_y

def blended_loss(pred_xy_norm, target_xy_norm, target_latlon, geo_stats, meter_weight=0.07):
    smooth_l1 = nn.functional.huber_loss(pred_xy_norm, target_xy_norm, delta=1.0)
    pred_x, pred_y = unnormalize_xy(pred_xy_norm, geo_stats)
    pred_lat, pred_lon = local_xy_to_latlon_torch(pred_x, pred_y, geo_stats.origin_lat, geo_stats.origin_lon)
    haversine = haversine_distance_torch(pred_lat, pred_lon, target_latlon[:, 0], target_latlon[:, 1]).mean()
    return smooth_l1 + meter_weight * (haversine / 100.0), haversine

## Evaluation Metrics

Model performance is evaluated using average Haversine distance in meters.

In [ ]:
def evaluate(model, data_loader, geo_stats):
    model.eval()

    total_haversine = 0.0
    total_count = 0

    with torch.no_grad():
        for images, _, latlon in data_loader:

            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)

            outputs = model(images)

            pred_x, pred_y = unnormalize_xy(outputs, geo_stats)
            pred_lat, pred_lon = local_xy_to_latlon_torch(pred_x, pred_y, geo_stats.origin_lat, geo_stats.origin_lon)
            pred_lat_np = pred_lat.cpu().numpy()
            pred_lon_np = pred_lon.cpu().numpy()

            target_lat_np = latlon[:, 0].cpu().numpy()
            target_lon_np = latlon[:, 1].cpu().numpy()

            distances = haversine_distance_np(target_lat_np, target_lon_np, pred_lat_np, pred_lon_np)

            total_haversine += float(distances.sum())
            total_count += len(distances)

    return total_haversine / max(total_count, 1)

## Dataset Loading and Train/Validation/Test Split

The Hugging Face dataset is loaded and split into:
- 70% training
- 15% validation
- 15% testing

Geographic normalization statistics are computed using only the training set.

In [ ]:
train_transform, eval_transform = build_transforms(IMAGE_SIZE)

# Load full HF dataset
full = load_dataset(DATASET_REPO)["train"]

# 70/15/15 split
split1 = full.train_test_split(
    test_size=0.15,
    seed=SEED
)

temp = split1["train"].train_test_split(
    test_size=0.1765,
    seed=SEED
)

dataset_train = temp["train"]
dataset_val   = temp["test"]
dataset_test  = split1["test"]

# geo stats only from training split
geo_stats = compute_geo_stats_from_arrays(
    dataset_train["latitude"],
    dataset_train["longitude"]
)

print(asdict(geo_stats))

## Dataset Overview

The dataset contains 1,115 geotagged campus images collected around the University of Pennsylvania.

Each image is paired with:
- latitude
- longitude

Images were collected from multiple campus locations under varying:
- lighting conditions
- weather conditions
- camera angles
- distances

The dataset was uploaded to Hugging Face for reproducibility and easier loading during training.

## Dataset Construction

The dataset objects cache resized images in memory and apply augmentations during training.

In [ ]:
train_dataset = GPSImageDataset(
    dataset_train,
    geo_stats,
    train_transform
)

val_dataset = GPSImageDataset(
    dataset_val,
    geo_stats,
    eval_transform
)

test_dataset = GPSImageDataset(
    dataset_test,
    geo_stats,
    eval_transform
)

## DataLoaders

PyTorch DataLoaders are used for efficient batching, prefetching, and parallel data loading.

In [ ]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

## Initialize Model and Optimizer

The model is initialized with:
- pretrained ImageNet weights
- partial backbone freezing
- AdamW optimizer
- cosine annealing learning rate schedule
- mixed precision training

In [ ]:
model = Model().to(device)
model = model.to(memory_format=torch.channels_last)

# Freeze all parameters
for p in model.model.parameters():
    p.requires_grad = False

# Unfreeze Layer4
for p in model.model.layer4.parameters():
    p.requires_grad = True

# Unfreeze regression head
for p in model.model.fc.parameters():
    p.requires_grad = True

# Compile model
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

scaler = torch.amp.GradScaler(
    enabled=device.type == "cuda"
)

## Transfer Learning Strategy

Training was performed in two stages:

1. Freeze most backbone layers and train only:
   - Layer4
   - regression head

2. Unfreeze the full network for fine-tuning at a lower learning rate.

This improves training stability and prevents catastrophic forgetting early in training.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_val_haversine = float("inf")
history = []

patience = 8
epochs_no_improve = 0
for epoch in range(1, EPOCHS + 1):
    if epoch == 3:
        print("Unfreezing full backbone...")

        for p in model.model.parameters():
            p.requires_grad=True

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr = LR/30,
            weight_decay=WEIGHT_DECAY
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS - epoch + 1
        )

    model.train()
    total_loss = 0.0
    total_count = 0

    for images, targets_xy, targets_latlon in train_dataloader:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        targets_xy = targets_xy.to(device)
        targets_latlon = targets_latlon.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda"):
            outputs = model(images)
            loss, batch_haversine = blended_loss(outputs, targets_xy, targets_latlon, geo_stats, METER_WEIGHT)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        total_loss += float(loss.item()) * batch_size
        total_count += batch_size

    scheduler.step()
    train_loss = total_loss / max(total_count, 1)
    if epoch % 10 == 0:
        train_haversine = evaluate(model, train_dataloader, geo_stats)
    else:
        train_haversine = 0
    val_haversine = evaluate(model, val_dataloader, geo_stats)

    result = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_mean_haversine_m": train_haversine,
        "val_mean_haversine_m": val_haversine,
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(result)

    print(
        f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
        f"train_haversine={train_haversine:.2f} m | val_haversine={val_haversine:.2f} m"
    )

    if val_haversine < best_val_haversine:
        best_val_haversine = val_haversine
        epochs_no_improve = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "geo_stats": asdict(geo_stats),
                "history": history,
                "backbone": BACKBONE,
                "image_size": IMAGE_SIZE,
            },
            OUTPUT_DIR / "best_model.pt",
        )
        with open(OUTPUT_DIR / "training_history.json", "w") as f:
            json.dump(history, f, indent=2)
        print(f"Saved new best model: {best_val_haversine:.2f} m")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

## Final Test Evaluation

Evaluate the best model checkpoint on the held-out test set.

In [ ]:
test_haversine = evaluate(
    model,
    test_dataloader,
    geo_stats
)

print(
    f"Final Test Haversine: "
    f"{test_haversine:.2f} meters"
)

print(
    f"Best validation Haversine distance: "
    f"{best_val_haversine:.2f} meters"
)

print(
    f"Artifacts saved to: "
    f"{OUTPUT_DIR.resolve()}"
)

## Export Submission Weights

The checkpoint is cleaned and exported into a lightweight `model.pt` file compatible with the leaderboard evaluation backend.

In [ ]:
checkpoint = torch.load(OUTPUT_DIR / "best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
state_dict = checkpoint["model_state_dict"]

clean_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("_orig_mod."):
        k = k[len("_orig_mod."):]

    if k.startswith("model."):
        k = k[len("model."):]

    clean_state_dict[k] = v

torch.save(clean_state_dict, "model.pt")
print("Saved clean model.pt")

## Final Results

| Metric | Result |
|---|---|
| Best Validation Distance | 26.61 m |
| Final Test Distance | 28.07 m |
| Backbone | ResNet50 |
| Image Size | 384 |
| Batch Size | 32 |

## Observations and Findings

Several trends were observed during experimentation:

- Stronger augmentations improved leaderboard generalization
- Smaller learning rates during fine-tuning improved stability
- Models with lower internal validation distance did not always perform best on the leaderboard
- Overfitting to the local validation split was a major challenge

The final model prioritized robustness and generalization over minimizing training error.